# 03 - Train one run and evaluate it (run owners: AW=A, GW=B, NG=C, NC=D, PN=E)

**Plain words:** every run uses the *same* script and the *same* split. Only the config file differs, so any difference in results is caused by the one thing that changed.

| Run | Model | Fakes from | Augmentation | Question |
|---|---|---|---|---|
| A | small CNN from scratch | one generator | off | baseline |
| B | ResNet-50 (transfer learning) | one generator | off | does transfer learning help? |
| C | ResNet-50 | one generator | on | does augmentation recover robustness? |
| D | ResNet-50 | several generators | off | does generator diversity help on unseen generators? |
| E | ResNet-50 | several generators | on | do the two effects combine? |

**Before you start:** GW has run notebook 02 and `audit_outputs/split.csv` exists on Drive.
**Use a GPU:** Runtime > Change runtime type > T4 GPU. (About 10 to 20 minutes per run.)

In [ ]:
# CELL 1 - run this first. It downloads the project code into this Colab session.
import os, subprocess, sys
REPO_URL = "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git"
CODE = "/content/repo"
if not os.path.isdir(CODE + "/src"):
    for branch in ("main", "gw/lean-pipeline"):          # main first; the branch is only a fallback
        subprocess.run(["rm", "-rf", CODE])
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, CODE])
        if r.returncode == 0 and os.path.exists(CODE + "/src/colab_setup.py"):
            print("Got the code from branch:", branch)
            break
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"])
sys.path.insert(0, CODE + "/src")
print("Code ready.")


In [ ]:
# CELL 2 - EDIT THESE TWO LINES
RUN = "B"          # A, B, C, D or E  (only the run you own)
MEMBER = "GW"      # your code

In [ ]:
# CELL 3 - connect, check GPU, find the split
import os, json, subprocess, torch
from colab_setup import setup
ROOT, REPO = setup()
SPLIT = os.path.join(ROOT, "audit_outputs", "split.csv")
assert os.path.exists(SPLIT), "split.csv not found: ask GW to run notebook 02 first"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (training will be very slow; change the runtime type)")
print(json.load(open(os.path.join(ROOT, "audit_outputs", "recommended.json"))))

In [ ]:
# CELL 4 - SMOKE TEST (about 1 minute). Proves the code works on your session before the long run.
!pip -q install pyyaml
!python {REPO}/src/train.py --config {REPO}/configs/run_{RUN}.yaml --split {SPLIT} --root {ROOT} --out /content/smoke --member {MEMBER} --smoke --workers 2

In [ ]:
# CELL 5 - THE REAL RUN. Prints one line per epoch. Keep this tab open; if Colab disconnects, run CELLS 1 to 3 and 5 again.
!python {REPO}/src/train.py --config {REPO}/configs/run_{RUN}.yaml --split {SPLIT} --root {ROOT} --out {ROOT}/reports --member {MEMBER} --workers 2

In [ ]:
# CELL 6 - look at the results
from IPython.display import Image, display
import pandas as pd
RUN_DIR = f"{ROOT}/reports/run_{RUN}"
display(Image(f"{RUN_DIR}/training_curves.png"))
display(Image(f"{RUN_DIR}/confusion_matrix.png"))
m = json.load(open(f"{RUN_DIR}/metrics.json"))
print(json.dumps(m["_run"], indent=1))
display(pd.read_csv(f"{ROOT}/reports/experiment_log.csv"))
print(open(f"{RUN_DIR}/shortcut_check.csv").read())

## Error analysis (this is marked: do not skip)

Open `failures.csv` and the pictures in `failures/`. Look at **at least 20** wrong predictions. Sort them into categories
(for example: heavily compressed, unusual scene, text in the image, a generator the model never saw, very smooth real photo).
Write the counts and what you think causes each. The next cell saves your notes into the run folder.

In [ ]:
# CELL 7 - look at the worst failures (most confident wrong answers)
import glob
from PIL import Image as PI
import matplotlib.pyplot as plt
fs = sorted(glob.glob(f"{RUN_DIR}/failures/*.png"))[:12]
fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for a, f in zip(ax.ravel(), fs):
    a.imshow(PI.open(f)); a.set_title(os.path.basename(f)[:34], fontsize=7); a.axis("off")
plt.show()
display(pd.read_csv(f"{RUN_DIR}/failures.csv").head(20))

In [ ]:
# CELL 8 - WRITE YOUR ERROR ANALYSIS here, then run the cell (edit the text, keep the structure)
ANALYSIS = '''
# Error analysis, Run {run}, by {member}

## What the numbers say
(2 sentences: seen vs unseen generators, and in the wild)

## Failure categories (at least 20 wrong predictions)
| Category | Count | Example image ids | Likely cause |
|---|---|---|---|
| e.g. heavily compressed real photos called AI | 7 | ... | compression removes fine detail |

## Most dangerous error for a user
(real called AI, or AI called real? who is harmed?)

## What I would try next (one testable idea)
'''.format(run=RUN, member=MEMBER)
open(f"{RUN_DIR}/analysis.md", "w").write(ANALYSIS)
print("saved", f"{RUN_DIR}/analysis.md")

## Share with the group (GitHub, no weights, no images)
The next cell zips the small result files (numbers, tables, charts). Upload the zip's contents to GitHub: `reports/run_<X>/`.

In [ ]:
# CELL 9 - download the small files for GitHub
import shutil, glob
from google.colab import files
small = [f for f in glob.glob(f"{RUN_DIR}/*") if os.path.isfile(f) and not f.endswith(".pt")]
os.makedirs(f"/content/upload_run_{RUN}", exist_ok=True)
for f in small:
    shutil.copy(f, f"/content/upload_run_{RUN}")
shutil.copy(f"{ROOT}/reports/experiment_log.csv", f"/content/upload_run_{RUN}")
shutil.make_archive(f"/content/run_{RUN}_results", "zip", f"/content/upload_run_{RUN}")
files.download(f"/content/run_{RUN}_results.zip")

## GW only: choose the final model
After A to E are done, GW compares them (validation numbers first, never the campus set) and picks one. Then run the next cell once.

In [ ]:
# CELL 10 (GW ONLY) - freeze the final model for the web app
FINAL_RUN = "run_E"      # change to the run you chose; write the reason in docs/final_model.md
import shutil, json
os.makedirs(f"{ROOT}/models", exist_ok=True)
shutil.copy(f"{ROOT}/reports/{FINAL_RUN}/best.pt", f"{ROOT}/models/final.pt")
thr = json.load(open(f"{ROOT}/reports/{FINAL_RUN}/metrics.json"))["_run"]["threshold"]
json.dump({"run": FINAL_RUN, "threshold": thr}, open(f"{ROOT}/models/final.json", "w"))
print("final.pt and final.json are in Drive/models. Threshold:", thr)